# 실습 — YOLO11 fine-tuning

이번 실습에서는 아프리카 야생동물 데이터셋 (4클래스 · 1,504장)으로 YOLO11 을 fine-tuning 하고, 학습 전과 후를 비교합니다.

| STEP | 하는 일 |
|---|---|
| 0 | 환경 준비 |
| 1 | 데이터셋 내려받고 눈으로 확인 (**TODO 1**) |
| 2 | `data.yaml` — 모델에게 줄 데이터 지도 (**TODO 2**) |
| 3 | 학습 **전에** 먼저 잰다 — 사전학습 모델의 기준선 ★ |
| 4 | fine-tuning (**TODO 3**) |
| 5 | 같은 잣대로, 숫자로 (**TODO 4**) |
| 6 | 달라진 점을 눈으로 (**TODO 5**) |

**시작 전에 반드시** — 메뉴 `런타임 → 런타임 유형 변경 → T4 GPU` 를 선택하세요.
STEP 4 의 학습이 T4 로는 12~18분, CPU 로는 몇 시간이 걸립니다.

## STEP 0 · 환경 준비

In [ ]:
!nvidia-smi

`ultralytics` 한 패키지면 충분합니다. 30초 정도 걸립니다.

In [ ]:
!pip install -q ultralytics

그래프 라벨에 한글이 깨지지 않도록 폰트를 설치합니다. 실패해도 실습에는 지장이 없습니다.

In [ ]:
# 한글 폰트 (약 20초) — 실패해도 그냥 넘어갑니다
try:
    !apt-get install -qq -y fonts-nanum > /dev/null
    import matplotlib.font_manager as fm
    fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
    import matplotlib.pyplot as plt
    plt.rcParams["font.family"] = "NanumGothic"
    plt.rcParams["axes.unicode_minus"] = False
    print("한글 폰트 준비 완료")
except Exception as e:
    print("폰트 설치 건너뜀:", e)

In [ ]:
import os, glob, random, zipfile, urllib.request

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ultralytics import YOLO

plt.rcParams["figure.dpi"] = 110
random.seed(0)

print("준비 완료")

## STEP 1 · 데이터셋 — African Wildlife

Ultralytics 가 공개한 **African Wildlife** 데이터셋을 씁니다. 아프리카 야생동물 4종의 사진에
바운딩 박스가 달려 있습니다.

| | |
|---|---|
| 클래스 | `buffalo` · `elephant` · `rhino` · `zebra` — 4개 |
| 규모 | train 1,052 / val 225 / test 227 — 총 1,504장 |
| 형식 | YOLO 표준 형식 (아래에서 직접 확인합니다) |
| 출처 | Ultralytics · AGPL-3.0 · docs.ultralytics.com/datasets/detect/african-wildlife |

이 데이터셋을 고른 이유가 있습니다. **elephant 와 zebra 는 COCO 80클래스에 있지만,
buffalo 와 rhino 는 없습니다.** "아는 물체 vs 모르는 물체" 앞에서 사전학습 모델이 어떻게
다르게 행동하는지, fine-tuning 이 정확히 무엇을 바꾸는지가 극적으로 보입니다.

아래 셀은 강사가 나눠 준 오프라인 배포본(`african_wildlife_mini.zip` · 340장 축소판)이
왼쪽 파일 탭에 업로드되어 있으면 그것을 쓰고, 없으면 원본(105 MB)을 GitHub 에서
내려받습니다. Colab 에서 10~20초면 끝납니다.

In [ ]:
URL  = "https://github.com/ultralytics/assets/releases/download/v0.0.0/african-wildlife.zip"
ZIP  = "african-wildlife.zip"
MINI = "african_wildlife_mini.zip"     # 오프라인 배포본 (강사 제공)
DATA = "african-wildlife"              # 압축을 풀 폴더

if not os.path.isdir(DATA):
    if os.path.exists(MINI):           # 1순위 — 나눠 받은 zip 을 업로드한 경우
        src = MINI
    else:                              # 2순위 — 원본을 직접 내려받기
        if not os.path.exists(ZIP):
            print("내려받는 중 ... (105 MB)")
            urllib.request.urlretrieve(URL, ZIP)
        src = ZIP
    with zipfile.ZipFile(src) as z:
        z.extractall(DATA)
    print("압축 해제 완료:", src, "→", DATA + "/")
else:
    print("이미 준비되어 있습니다:", DATA + "/")

In [ ]:
# 폴더 구조 — images 와 labels 가 같은 파일명으로 짝을 이룹니다
for split in ["train", "val", "test"]:
    n_img = len(glob.glob(f"{DATA}/images/{split}/*.jpg"))
    n_lab = len(glob.glob(f"{DATA}/labels/{split}/*.txt"))
    print(f"{split:<6} 이미지 {n_img:>5}장   라벨 {n_lab:>5}개")
# 원본이면 1052 / 225 / 227, 미니 배포본이면 248 / 60 / 32 이 나와야 합니다

### 라벨 파일 — YOLO 형식

이미지 한 장마다 **같은 이름의 텍스트 파일**이 하나씩 있고, 한 줄이 박스 하나입니다.

```
<클래스 번호> <중심 x> <중심 y> <너비> <높이>
```

좌표 네 개는 **이미지 크기로 나눈 0~1 정규화 값**입니다. 추론 결과에서 쓰는
`xyxy`(픽셀 단위 · 모서리 두 점)와 다릅니다 — 정규화해 두면 이미지를 줄이거나 키워도
라벨을 고칠 필요가 없기 때문입니다.

| 클래스 번호 | 이름 |
|---|---|
| 0 | buffalo (아프리카물소) |
| 1 | elephant (코끼리) |
| 2 | rhino (코뿔소) |
| 3 | zebra (얼룩말) |

In [ ]:
CLASSES = {0: "buffalo", 1: "elephant", 2: "rhino", 3: "zebra"}

lab = sorted(glob.glob(f"{DATA}/labels/train/*.txt"))[0]
print(lab)
print(open(lab).read())

### TODO 1 — 정규화 좌표를 픽셀 좌표로

중심·크기 형식 `(cx, cy, bw, bh)` 를 모서리 형식 `(x1, y1, x2, y2)` 로 바꾸는 함수를 완성하세요.
중심에서 크기의 절반을 빼면 왼쪽(위) 모서리, 더하면 오른쪽(아래) 모서리입니다. 마지막에
이미지 너비·높이를 곱해 픽셀 단위로 만듭니다. 아래의 자기 검증이 `통과!` 를 출력하면 맞은 것입니다.

In [ ]:
def yolo_to_xyxy(cx, cy, bw, bh, w, h):
    """정규화 (cx, cy, bw, bh)  →  픽셀 (x1, y1, x2, y2)"""
    # TODO 1 ── 아래 네 줄을 채우세요
    x1 = ...          # (중심 x − 너비/2) × 이미지 너비
    y1 = ...          # (중심 y − 높이/2) × 이미지 높이
    x2 = ...          # (중심 x + 너비/2) × 이미지 너비
    y2 = ...          # (중심 y + 높이/2) × 이미지 높이
    return x1, y1, x2, y2


# 자기 검증 — 틀리면 AssertionError 가 납니다
assert yolo_to_xyxy(0.5, 0.5, 0.5, 0.5, 640, 480) == (160.0, 120.0, 480.0, 360.0)
assert yolo_to_xyxy(0.25, 0.5, 0.5, 1.0, 400, 200) == (0.0, 0.0, 200.0, 200.0)
print("통과!")

In [ ]:
# 클래스별 대표 이미지에 정답 라벨을 그려 봅니다 (그대로 실행)
DEMO = {0: "1 (1).jpg", 1: "2 (1).jpg", 2: "3 (106).jpg", 3: "4 (10).jpg"}

def demo_path(c, split="val"):
    """클래스 c 의 대표 이미지 경로. 없으면 그 클래스만 들어 있는 첫 이미지로 대체."""
    p = f"{DATA}/images/{split}/{DEMO[c]}"
    if os.path.exists(p):
        return p
    for lp in sorted(glob.glob(f"{DATA}/labels/{split}/*.txt")):
        if {int(l.split()[0]) for l in open(lp) if l.strip()} == {c}:
            return lp.replace("labels", "images").replace(".txt", ".jpg")

fig, axes = plt.subplots(1, 4, figsize=(17, 4))
for ax, c in zip(axes, CLASSES):
    p = demo_path(c)
    img = cv2.imread(p)
    h, w = img.shape[:2]
    for line in open(p.replace("images", "labels").replace(".jpg", ".txt")):
        cid, cx, cy, bw, bh = line.split()
        x1, y1, x2, y2 = map(int, yolo_to_xyxy(float(cx), float(cy), float(bw), float(bh), w, h))
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 210, 0), max(2, w // 300))
    ax.imshow(img[:, :, ::-1])
    ax.set_title(f"{c} · {CLASSES[c]}", fontsize=13)
    ax.axis("off")
plt.tight_layout(); plt.show()

초록 박스가 동물을 잘 감싸고 있으면 라벨이 제대로 읽힌 것입니다. **모델은 이 박스를 교과서
삼아 배웁니다** — 라벨이 틀리면 모델도 틀리게 배웁니다. 참고로 이 데이터셋에도 라벨이 잘못 달린
사진이 몇 장 섞여 있습니다(공개 데이터셋의 현실입니다). 뒤에서 결과를 해석할 때 기억해 두세요.

## STEP 2 · `data.yaml` — 모델에게 줄 데이터 지도

`model.train()` 은 데이터가 어디 있고 클래스가 무엇인지 **`data.yaml` 파일 하나로** 전달받습니다.
필요한 항목은 네 가지뿐입니다.

| 항목 | 뜻 |
|---|---|
| `path` | 데이터셋 뿌리 폴더 (절대경로 권장) |
| `train` | 학습 이미지 폴더 — `path` 기준 상대경로 |
| `val` | 검증 이미지 폴더 — `path` 기준 상대경로 |
| `names` | 클래스 번호 → 이름 |

라벨 폴더는 적지 않습니다 — YOLO 가 경로의 `images` 를 `labels` 로 바꿔 스스로 찾습니다.

### TODO 2 — data.yaml 완성

`train` / `val` 경로 두 곳과 클래스 이름 네 개를 채우세요. STEP 1 에서 확인한 폴더 구조와
클래스 표가 그대로 답입니다.

In [ ]:
yaml_text = f"""\
path: {os.path.abspath(DATA)}

# TODO 2 ── train / val 경로와 클래스 이름 4개를 채우세요
train: ...
val: ...

names:
  0: ...
  1: ...
  2: ...
  3: ...
"""

with open("data.yaml", "w") as f:
    f.write(yaml_text)
print(yaml_text)

In [ ]:
# data.yaml 검사 — 여기서 틀리면 뒤가 전부 무의미하므로 반드시 "통과!" 를 보고 넘어가세요
import yaml

cfg = yaml.safe_load(open("data.yaml"))
ok = True

for k in ["path", "train", "val", "names"]:
    if k not in cfg or cfg[k] in (None, "..."):
        print("채우지 않음:", k); ok = False

if ok:
    for k in ["train", "val"]:
        d = os.path.join(cfg["path"], str(cfg[k]))
        n = len(glob.glob(os.path.join(d, "*.jpg")))
        print(f"{k:<5} → {d}   ({n}장)")
        if n == 0:
            print("   ⚠ 이미지가 0장 — 경로 오타를 확인하세요"); ok = False
    names = cfg["names"]
    print("names →", names)
    if len(names) != 4 or any(v in (None, "...") for v in names.values()):
        print("   ⚠ 클래스 이름 4개를 모두 채워야 합니다"); ok = False

print("\n통과!" if ok else "\n위 항목을 고치고 이 셀을 다시 실행하세요.")

## STEP 3 · 학습 전에 먼저 잰다 ★

무언가를 바꾸기 전에는 **바꾸기 전 상태를 먼저 재 둬야** 바뀐 양을 말할 수 있습니다.
fine-tuning 도 같습니다 — COCO 사전학습 YOLO11 이 이 데이터에서 지금 어떻게 행동하는지부터
기록합니다. 이 기록이 오늘의 **기준선(baseline)** 이 됩니다.

In [ ]:
coco = YOLO("yolo11n.pt")          # COCO 80클래스로 사전학습된 모델

animals = {i: coco.names[i] for i in range(14, 24)}     # COCO 어휘 중 동물 구간
print("COCO 어휘 중 동물:", animals)
print()
for want in CLASSES.values():
    있음 = want in coco.names.values()
    print(f"  {want:<9} → COCO 어휘에 {'있음 ✓' if 있음 else '없음 ✗'}")

`buffalo` 와 `rhino` 는 어휘에 없습니다. 그러면 사전학습 모델은 버팔로 사진 앞에서 어떻게
할까요 — 못 본 척할까요, 아는 단어 중 가장 비슷한 것으로 부를까요? 직접 확인합니다.

In [ ]:
# 대표 이미지 4장을 사전학습 모델로 추론합니다 (그대로 실행)
fig, axes = plt.subplots(1, 4, figsize=(17, 4))
for ax, c in zip(axes, CLASSES):
    r = coco(demo_path(c), conf=0.25, verbose=False)[0]
    ax.imshow(r.plot()[:, :, ::-1])
    ax.set_title(f"정답: {CLASSES[c]}", fontsize=13)
    ax.axis("off")
plt.suptitle("사전학습 YOLO11n 이 붙인 이름", y=1.03, fontsize=14)
plt.tight_layout(); plt.show()

**elephant 와 zebra 는 맞히고, buffalo 는 `cow`, rhino 는 `cow` 나 `elephant` 로 부릅니다.**
모델이 멍청한 것이 아닙니다 — 위치는 정확하게 찾았는데 **부를 이름이 어휘에 없어서**, 아는
단어 중 가장 비슷한 것을 골랐을 뿐입니다. 네 장으로 끝내지 말고 12장으로 표를 만들어 봅니다.

In [ ]:
# 클래스별 val 이미지 3장씩 — 각 정답 박스와 가장 많이 겹치는 예측의 이름을 기록 (그대로 실행)
def iou(a, b):
    """두 xyxy 박스의 IoU"""
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union else 0

def first_class(lp):
    line = open(lp).readline().split()
    return int(line[0]) if line else -1

rows = []
for c in CLASSES:
    picked = [lp for lp in sorted(glob.glob(f"{DATA}/labels/val/*.txt")) if first_class(lp) == c][:3]
    for lp in picked:
        ip = lp.replace("labels", "images").replace(".txt", ".jpg")
        h, w = cv2.imread(ip).shape[:2]
        r = coco(ip, conf=0.25, verbose=False)[0]
        preds = [(r.names[int(b.cls)], b.xyxy[0].tolist()) for b in r.boxes]
        for line in open(lp):
            cid, cx, cy, bw, bh = line.split()
            g = yolo_to_xyxy(float(cx), float(cy), float(bw), float(bh), w, h)
            name, best_iou = "(못 찾음)", 0.5            # IoU 0.5 이상 겹쳐야 인정
            for pn, pb in preds:
                v = iou(g, pb)
                if v >= best_iou:
                    name, best_iou = pn, v
            rows.append({"정답": CLASSES[int(cid)], "사전학습 모델의 호칭": name})

df_match = pd.DataFrame(rows)
pd.crosstab(df_match["정답"], df_match["사전학습 모델의 호칭"])

표를 읽어 보세요. 정답이 buffalo·rhino 인 줄에는 올바른 이름이 하나도 없습니다 —
**이 모델로는 threshold 를 아무리 조정해도 buffalo 를 buffalo 라고 부르게 만들 수 없습니다.**
어휘 자체를 바꿔야 하고, 그것이 fine-tuning 입니다. 이 표는 STEP 5 에서 다시 만납니다.

## STEP 4 · fine-tuning

`model.train()` 한 줄이지만, 안에서는 이런 일이 일어납니다.

| 부분 | 무슨 일이 일어나나 |
|---|---|
| backbone · neck (특징 추출) | COCO 에서 배운 가중치를 **그대로 이어받아** 이 데이터로 미세 조정 |
| head (클래스·박스 출력) | 80클래스용을 떼어 내고 **4클래스용을 새로 달아** 학습 |

"모서리·질감·형태를 보는 눈"은 재사용하고 "이름을 붙이는 입"만 새로 배우는 셈입니다.
그래서 1,000여 장으로도 — 밑바닥부터 배우려면 어림없는 양으로도 — 쓸 만한 탐지기가 나옵니다.

### TODO 3 — `train()` 호출 완성

데이터 지도, epoch 수, 입력 크기 세 인자를 채우세요. T4 기준 20 epoch 에 **12~18분**입니다.
시간이 빠듯하면 `EPOCHS = 10` — mAP 는 조금 낮아지지만 흐름은 같습니다.

In [ ]:
EPOCHS = 20          # ★ 시간이 빠듯하면 10 으로

model = YOLO("yolo11n.pt")               # 사전학습 가중치에서 출발합니다
res = model.train(
    # TODO 3 ── 세 인자를 채우세요
    data=...,                            # 데이터 지도 (STEP 2 에서 만든 파일 이름, 문자열)
    epochs=...,                          # 전체 데이터를 몇 바퀴 돌까 (위의 EPOCHS)
    imgsz=...,                           # 입력 이미지 크기 (추론 때 기본값과 같은 640)
    batch=16,
    seed=0,
)

기다리는 동안 로그를 읽어 봅니다.

- 시작 부분의 `Overriding model.yaml nc=80 with nc=4` — head 를 4클래스용으로 바꿔 달았다는
  뜻입니다. 그 근처에 `Remapped 2/4 cls head rows ...` 가 보인다면, 이름이 같은
  elephant·zebra 는 head 가중치까지 이어받았다는 뜻입니다.
- 매 epoch 줄: `box_loss`(박스 위치) · `cls_loss`(클래스 이름) · `dfl_loss`(박스 경계)
  세 손실이 내려가는지 보세요.
- 매 epoch 끝의 `mAP50` 이 올라가다가 **평평해지는지**도 — 평평해진 뒤로는 epoch 을 늘려도
  얻는 것이 별로 없습니다.

In [ ]:
SAVE_DIR = str(res.save_dir)
BEST = os.path.join(SAVE_DIR, "weights", "best.pt")
print("결과 폴더  :", SAVE_DIR)
print("최고 가중치:", BEST, "| 존재:", os.path.exists(BEST))

plt.figure(figsize=(14, 7))
plt.imshow(plt.imread(os.path.join(SAVE_DIR, "results.png")))
plt.axis("off")
plt.title("학습 곡선 — 왼쪽: 손실들 · 오른쪽: 정밀도/재현율/mAP", fontsize=12)
plt.show()

영상 분류에서 하던 과적합 진단이 그대로 통합니다 — train 손실은 내려가는데 **val 손실이
도로 올라가기 시작하면** 과적합입니다. `best.pt` 는 **val 성적이 가장 좋았던 시점**의 가중치라
과적합 구간을 자동으로 피해 줍니다(마지막 epoch 의 `last.pt` 와 다른 점입니다).

## STEP 5 · 같은 잣대로, 숫자로 ★★

val 셋(학습에 쓰지 않은 데이터)에서 성적표를 뽑습니다. 두 숫자만 기억하면 됩니다.

| 지표 | 뜻 |
|---|---|
| `mAP50` | 예측 박스가 정답과 IoU ≥ 0.5 면 정답으로 치는, 후한 채점 |
| `mAP50-95` | IoU 기준을 0.5→0.95 로 올려 가며 평균 — 박스가 얼마나 **정밀한지**까지 보는 채점 |

In [ ]:
best = YOLO(BEST)
m = best.val(data="data.yaml")     # val 셋 전체를 채점합니다 — 클래스별 표가 출력됩니다

### TODO 4 — 성적표 꺼내기

값은 `m.box` 안에 있습니다: 전체 mAP50 은 `m.box.map50`, 전체 mAP50-95 는 `m.box.map`,
클래스별 AP50-95 배열(클래스 번호 순)은 `m.box.maps` 입니다.

In [ ]:
# TODO 4 ── 세 값을 꺼내세요
map50   = ...                    # 전체 mAP50
map5095 = ...                    # 전체 mAP50-95
per_cls = ...                    # 클래스별 AP50-95 배열

print(f"mAP50    : {map50:.3f}")
print(f"mAP50-95 : {map5095:.3f}")

# 학습 전(STEP 3 표)과 후를 한 표로 — 클래스별 train 박스 수도 함께 (아래는 그대로 실행)
n_train = {c: 0 for c in CLASSES}
for lp in glob.glob(f"{DATA}/labels/train/*.txt"):
    for line in open(lp):
        if line.strip():
            n_train[int(line.split()[0])] += 1

pre = df_match.groupby("정답")["사전학습 모델의 호칭"].agg(lambda s: s.mode()[0])
pd.DataFrame({
    "클래스":            [CLASSES[c] for c in CLASSES],
    "train 박스 수":     [n_train[c] for c in CLASSES],
    "학습 전 모델의 호칭": [pre[CLASSES[c]] for c in CLASSES],
    "학습 후 AP50-95":   [round(float(per_cls[c]), 3) for c in CLASSES],
})

In [ ]:
# 어떤 클래스끼리 헷갈리는지 — 혼동 행렬 (그대로 실행)
cm = os.path.join(str(m.save_dir), "confusion_matrix_normalized.png")
plt.figure(figsize=(9, 8))
plt.imshow(plt.imread(cm))
plt.axis("off"); plt.show()

### 결과를 어떻게 읽을 것인가

| 관찰 | 해석 |
|---|---|
| mAP50 이 0.9 안팎 | 원본 데이터 · 20 epoch 에서 정상 범위입니다 (미니 배포본은 이보다 낮습니다) |
| 특정 클래스만 낮다 | 위 표에서 train 박스 수부터 확인 — 데이터 양이 곧 성능입니다 |
| 혼동 행렬의 buffalo↔rhino | 생김새가 비슷한 쌍 + 라벨 오류 몇 장의 영향입니다 |
| background 행/열이 크다 | 못 찾았거나(놓침) 없는 것을 찾은(오탐) 경우 — 작은 동물·가림이 대부분 |

숫자가 기대보다 낮아도 결론은 하나가 아닙니다 — epoch 부족(곡선이 아직 오르는 중이었나),
데이터 부족(클래스별 박스 수), 라벨 품질. **어느 쪽인지 근거를 대는 것**이 분석입니다.

## STEP 6 · 달라진 점을 눈으로

숫자는 요약일 뿐, 설득은 그림이 합니다. 학습·검증에 쓰지 않은 **test 이미지**를
학습 전 모델과 학습 후 모델에 나란히 통과시킵니다.

### TODO 5 — 두 모델로 같은 이미지 추론

왼쪽 열에는 사전학습 모델(`coco`), 오른쪽 열에는 방금 학습한 모델(`best`)의 결과를 넣으세요.
추론 문법은 두 모델이 완전히 같습니다.

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(13, 21))
os.makedirs("outputs", exist_ok=True)

for row, c in enumerate(CLASSES):
    p = demo_path(c, split="test")                    # test 셋에서 클래스별 한 장
    # TODO 5 ── 두 줄을 채우세요 (같은 이미지 p 를 두 모델에, conf=0.25)
    r_pre = ...                                       # 사전학습 모델(coco) 로 추론한 결과 [0]
    r_ft  = ...                                       # fine-tuning 모델(best) 로 추론한 결과 [0]

    for col, (r, name) in enumerate([(r_pre, "학습 전 (COCO 어휘)"), (r_ft, "학습 후 (4클래스)")]):
        axes[row][col].imshow(r.plot()[:, :, ::-1])
        axes[row][col].set_title(f"{CLASSES[c]} — {name}", fontsize=12)
        axes[row][col].axis("off")

plt.tight_layout()
plt.savefig("outputs/before_after.png", dpi=120, bbox_inches="tight")
plt.show()
print("저장: outputs/before_after.png")

왼쪽 열에서 `cow` 나 `elephant` 로 불리던 버팔로·코뿔소가 오른쪽 열에서는 제 이름으로 잡힙니다. 한 가지 더 —
**학습 후 모델은 이제 사람도 자동차도 못 찾습니다.** 어휘를 4개로 바꿨기 때문입니다.
fine-tuning 은 공짜가 아니라 **교환**입니다. 원래 어휘도 필요하면 사전학습 모델을 함께
쓰거나, 원래 데이터를 섞어 학습해야 합니다.

In [ ]:
# 내 사진으로도 — 동물원·사파리 사진이 있으면 가장 좋습니다 (Colab 전용 셀)
try:
    from google.colab import files
    up = files.upload()                     # 파일 선택 창이 뜹니다 — 건너뛰려면 취소
    my_imgs = list(up.keys())
except ImportError:
    my_imgs = []                            # 로컬 환경이면 ["사진.jpg"] 처럼 직접 적으세요

for p in my_imgs:
    fig, axes = plt.subplots(1, 2, figsize=(13, 6))
    for ax, (mdl, name) in zip(axes, [(coco, "학습 전"), (best, "학습 후")]):
        r = mdl(p, conf=0.25, verbose=False)[0]
        ax.imshow(r.plot()[:, :, ::-1])
        ax.set_title(f"{name} — {os.path.basename(p)}", fontsize=12)
        ax.axis("off")
    plt.tight_layout(); plt.show()

### 모델 저장과 배포 (선택 — 시간이 남으면)

`best.pt` 파일 하나가 오늘의 결과물입니다. 내려받아 두면 어디서든 `YOLO("best.pt")` 로
다시 쓸 수 있고, ONNX 로 바꾸면 PyTorch 가 없는 환경에서도 돌릴 수 있습니다.

In [ ]:
# best.pt 내려받기 + ONNX 변환 (처음 한 번은 변환용 패키지 설치로 30초쯤 걸립니다)
onnx_path = best.export(format="onnx")
print("ONNX 저장:", onnx_path)

try:
    from google.colab import files
    files.download(BEST)                    # best.pt  (약 5 MB)
    files.download(onnx_path)               # best.onnx (약 10 MB)
except ImportError:
    print("로컬 환경 — 파일 탐색기에서 직접 복사하세요")

## 정리

오늘 한 일을 한 줄씩 되짚으면 그대로 fine-tuning 의 표준 절차가 됩니다.

| 단계 | 오늘 한 일 | 내 데이터로 할 때 |
|---|---|---|
| 데이터 | 폴더 구조·라벨 형식을 눈으로 검증 | 같은 구조로 준비 — 라벨링 도구: Roboflow · CVAT · labelImg |
| data.yaml | 경로·클래스 지도 작성 | 경로와 `names` 만 바꾸면 이 노트북이 그대로 돌아갑니다 |
| 기준선 ★ | 사전학습 모델의 행동을 먼저 기록 | 항상 먼저 — 그래야 "얼마나 나아졌나"를 증명할 수 있습니다 |
| 학습 | `train()` — head 교체 + 미세 조정 | `epochs` · `imgsz` · `batch` 부터 조정 |
| 평가 | mAP · 혼동 행렬 · 학습 전후 나란히 | 숫자 하나가 아니라 근거를 갖춘 분석 |

**더 해 볼 것** — ① `EPOCHS` 를 늘리거나 모델을 `yolo11s.pt` 로 키워 mAP 이 얼마나 오르는지,
② `conf` 를 바꿔 가며 놓침/오탐이 어떻게 움직이는지, ③ 내 관심 물체를 직접 라벨링해
이 노트북에 그대로 태워 보기.

---

데이터: African Wildlife Dataset — Ultralytics 제공 (AGPL-3.0) ·
[docs.ultralytics.com/datasets/detect/african-wildlife](https://docs.ultralytics.com/datasets/detect/african-wildlife/)